<a href="https://colab.research.google.com/github/rahmangamer12/AI-Agents-Calculator/blob/main/03_Funciton_Calling_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
!pip install -Uq langchain google-generativeai python-dotenv langchain_google_genai

In [10]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Access the API key
GOOGLE_GEMINI_API_KEY = os.getenv("GOOGLE_GEMINI_API_KEY")



In [11]:
# Define a basic calculator tool
class Calculator:
    def calculate(self, expression: str) -> str:
        try:
            # Use eval to compute the result safely (restrict built-ins)
            result = eval(expression, {"__builtins__": None}, {})
            return f"The result is: {result}"
        except Exception as e:
            return f"Error: {str(e)}"


In [12]:
from langchain.tools import tool

# Create a LangChain tool for the calculator
@tool
def calculator(expression: str) -> str:
    """
    Perform arithmetic calculations.
    Input: A mathematical expression as a string (e.g., "2 + 2").
    Output: Result of the calculation as a string.
    """
    calc = Calculator()
    return calc.calculate(expression)


In [13]:
import google.generativeai as genai

# Configure the Gemini API key
genai.configure(api_key=GOOGLE_GEMINI_API_KEY)

def query_gemini(prompt: str) -> str:
    """
    Query the Google Gemini model with a natural language prompt.
    """
    response = genai.generate_text(
        model="gemini-1.5",  # Use Gemini 1.5 Flash
        prompt=prompt
    )
    return response.result  # Extract the text result


In [14]:
from langchain.agents import initialize_agent, Tool, AgentType
from langchain_google_genai import ChatGoogleGenerativeAI  # Use Google's Palm API (Gemini)
from langchain.tools import tool
import os

# Define a calculator function
@tool
def calculator(expression: str) -> str:
    """
    Perform arithmetic calculations.
    Input: A mathematical expression as a string (e.g., "2 + 2").
    Output: Result of the calculation as a string.
    """
    try:
        result = eval(expression, {"__builtins__": None}, {})
        return str(result)
    except Exception as e:
        return f"Error: {e}"

# Define the tools for the agent
tools = [
    Tool(name="Calculator", func=calculator, description="Performs calculations."),
]

# Load the API key from the environment
GOOGLE_GEMINI_API_KEY = os.getenv("GOOGLE_GEMINI_API_KEY")

# Initialize the Google Generative AI model (Gemini)
llm = ChatGoogleGenerativeAI(
    api_key=GOOGLE_GEMINI_API_KEY,
    model="gemini-1.5"  # Specify the model you want to use (e.g., gemini-1.5)
)

# Initialize the agent with the calculator tool
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
)

In [15]:
def handle_query(user_input: str):
    """
    Process user input and determine whether to use the calculator or Gemini.
    """
    if any(op in user_input for op in ['+', '-', '*', '/']):
        # Use the calculator tool for arithmetic operations
        result = calculator(user_input)
    else:
        # Use Google Gemini for natural language queries
        result = query_gemini(user_input)

    return result

In [16]:
# Test the system
print("Welcome to AI Agents Calcuiator")
while True:
    user_query = input("Enter your query (or 'exit' to quit): ")
    if user_query.lower() == 'exit':
        break

    response = handle_query(user_query)
    print(f"Response: {response}")

Welcome to AI Agents Calcuiator
Enter your query (or 'exit' to quit): 45+45
Response: 90
Enter your query (or 'exit' to quit): exit
